The goal of this sec


In [1]:
import random
import h5py
import numpy as np

For simplicity, we will generate the initial simple data with the following constrinats
- We will always order particle labels from least to most, i.e. pi.pj or ei.ej for j>i. 
- For ei.pj, the polarization will always be on the left regardless of the order
- We will generate a numerator of up to $r$ monomials over a single denominator which contains only momentum dot products, ensuring that the entire term has mass dimension $q$ and little group weight $\ell$. Recall that under a little group transformation $L^\mu_{~\nu}(k)k^\nu = k^\nu$, polarization vectors behave as $$L^\mu_{~\nu}(k)\epsilon_\lambda^\nu(k) = \sum_{\lambda\lambda'}D_{\lambda\lambda'}\epsilon_{\lambda'}^\nu(k) = e^{i\theta\lambda}\epsilon_{\lambda}^\nu(k)$$
where $\lambda$ is the helicity. We will generate amplitudes with various mass dimensions and little group weights in order to make the model as general as possible (allowing for dimensionalful couplings). It may be desireable to restrict ourselves to only renormalisible theories + gravity. Discuss.
-  The range of $r$, $q$ and $\ell$ should be specified as parameters

TODO:
- [] There should be an equal number of $\epsilon_i$'s in each term of the numerator, but there aren't. Should pre-decide the number of gluons/scalars/gravitons etc.
- [] Convert to sympy for easier manipulation

In [125]:
def random_pp(n):
    # p_i.p_j with i<j
    i = random.randint(1, n-1)
    j = random.randint(i+1, n)
    return f"p{i}.p{j}"

def random_ep(n,m,l): # we generate an n-particle e.p where e is in the range m to l
    # e_i.p_j with i ≠ j; always put e on the left.
    i = random.randint(m, l)
    j = random.choice([x for x in range(1, n+1) if x != i]) # i!=j
    return f"e{i}.p{j}"

def random_ee(n,m,l):
    # e_i.e_j with i<j
    i = random.randint(1, m)
    j = random.randint(m+1, l)
    return f"e{i}.e{j}"

def generate_monomial(n, n_gluons, n_gravitons, dim):
    """
    Generate a product of p.p factors, e.p factors, and e.e factors, with overall mass dimension dim.
    The factors are randomly chosen, shuffled, and multiplied.
    We need to include some number of gluons and gravitons in the expression. To be systematic about it, we will choose particles 1-n_gluons to be gluons, n_gluons+1 to n_gravitons to be gravitons, and the rest to be scalars.
    """
    factors = []
    # Determine how we will distribute the gluon polarizations: how many ei.ej or ei.pj?
    # Need to determine b,c.
    # Each gluon's polarization vector must appear once. An e_i.p factor uses one vector
    # (contributing little-group weight 1), while an e_i.e factor uses two (weight 2).
    # Hence, if we choose c pairs (e_i.e_j factors), then the remaining b vectors go into e_i.p factors,
    # with b + 2*c = n_gluons.
    max_pairs = n_gluons // 2
    c_gluons = random.randint(0, max_pairs)
    b_gluons = n_gluons - 2 * c_gluons
  
    # For gravitons, we do the same thing with the remaining n_gravitons.
    max_graviton_pairs = n_gravitons // 2
    c_graviton = random.randint(0, max_graviton_pairs)
    b_graviton = n_gravitons - 2 * c_graviton 

    if dim == 0: # Can't have a mass dimension of 0 with numerators involving momentum
        b_gluons = 0
        b_graviton = 0
    if dim == 1: # We can't have a mass dimension of 1 with only (massless) scalars, so we need at least one gluon. In general, we could just have an mass m here.
        b_gluons = 1
        b_graviton = 0
    if dim == 2: # Need to pick between 2 gluons or 1 graviton. Will will use random choice here, via the function 
        if random.choice([0,1]) == 0:
            b_gluons = 2
            b_graviton = 0
        else:
            b_gluons = 0
            b_graviton = 1
    # We assume that the overall mass dimension satisfies:
    #     dim = 0.5 * pp_factors + b_gluons + 0.5 b_gravitons (gravitons have to polarization vectors, i.e. dim[p.p] = dim[(e.p)^2])
    # so, solving for pp_factors gives:
    pp_factors = 2 * (dim - b_gluons) - b_graviton
    print(f"Trying to make a monomial with {n} particles, {n_gluons} gluons, {n_gravitons} gravitons, and mass dimension {dim}. Need {pp_factors} pp factors, {b_gluons} gluon e.p factors, {c_gluons} gluon e.e factors, {b_graviton} graviton e.p factors, and {c_graviton} graviton e.e factors.")
    for _ in range(pp_factors):
        factors.append(random_pp(n))
    # For gluonic ei.pi, each factor is of the form "e{i}.p{j}" with j != i.
    # Create and shuffle the list of gluon polarization indices.
    gluon_indices = list(range(1, n_gluons + 1))
    random.shuffle(gluon_indices)
    print(gluon_indices)
    # For the first b_gluons indices, form ei.pj factors (ensuring p index is not equal to i)
    for i in range(b_gluons):
        index = gluon_indices[i]
        available_momenta = [j for j in range(1, n+1) if j != index]
        momentum_index = random.choice(available_momenta)
        factors.append(f"e{index}.p{momentum_index}")
    
    # For the remaining 2*c_gluons indices, form e_i.e_j pairs.
    for i in range(c_gluons):
        idx1 = gluon_indices[b_gluons + 2*i]
        idx2 = gluon_indices[b_gluons + 2*i + 1]
        factors.append(f"e{idx1}.e{idx2}")
    
    # Create and shuffle the list of graviton polarization indices. We need to go from n_gluons+1 to n_gr
    graviton_indices = list(range(n_gluons+1, n_gluons + n_gravitons+1))
    random.shuffle(graviton_indices)
    print(graviton_indices)
    # For the first b_graviton indices, form ei.pj factors (ensuring p index is not equal to i)
    for i in range(b_graviton):
        index = graviton_indices[i]
        available_momenta = [j for j in range(1, n+1) if j != index]
        momentum_index1 = random.choice(available_momenta)
        momentum_index2 = random.choice(available_momenta)
        factors.append(f"e{index}.p{momentum_index1}")
        factors.append(f"e{index}.p{momentum_index2}")
    
    # For the remaining 2*c_graviton indices, form e_i.e_j pairs.
    for i in range(c_graviton):
        idx1 = graviton_indices[b_graviton + 2*i]
        idx2 = graviton_indices[b_graviton + 2*i + 1]
        factors.append(f"e{idx1}.e{idx2}")
        factors.append(f"e{idx1}.e{idx2}")
    # We won't generate terms of the form ei.pj ei.ek ek.pl. In tensorial form, these would come from pj^m*ei_{mn}ek^{nr}pl_r. Need to fix this!
    #random.shuffle(factors)
    return " * ".join(factors) if factors else "1"


In [126]:
print(generate_monomial(4,2,0,0))

Trying to make a monomial with 4 particles, 2 gluons, 0 gravitons, and mass dimension 0. Need 0 pp factors, 0 gluon e.p factors, 0 gluon e.e factors, 0 graviton e.p factors, and 0 graviton e.e factors.
[1, 2]
[]
1


In [127]:

def generate_denominator(n, d):
    """Generate a product of d momentum dot products."""
    factors = [random_pp(n) for _ in range(d)]
    return " * ".join(factors) if factors else "1"

def generate_amplitude(n_particles=4, r=3, q=4, ell=2, d=1):
    r"""
    Generate an amplitude-like expression in the form

         (x + y + ...)/D

    where each numerator term is generated as a product of
      - a factors p.p (each of mass-dimension 2),
      - b factors e.p (each of mass-dimension 1, little-group weight 1),
      - c factors e.e (dimensionless, little-group weight 2),
    with the constraints
         2a + b = q + 2d,   and   b + 2c = ell.
    
    We set T = (q+2d-ell)/2 and choose an integer c with 0 ≤ c ≤ floor(ell/2),
    then a = T + c and b = ell - 2c.
    
    The common denominator is generated once (as a product of d momentum dot products)
    and each numerator term appears with an overall sign.
    
    Parameters:
      - n_particles: number of external particles.
      - r: number of terms in the numerator sum.
      - q: target overall mass dimension.
      - ell: target little-group weight.
      - d: number of factors in the common denominator.
      
    Note: q+2*d must be ≥ ell and (q+2*d-ell) must be even.
    """
    if (q + 2*d - ell) < 0 or ((q + 2*d - ell) % 2 != 0):
        raise ValueError("Inconsistent parameters: require q+2*d>=ell and (q+2*d-ell) even.")
    
    T = (q + 2*d - ell) // 2
    common_den = generate_denominator(n_particles, d)
    
    numerator_terms = []
    for _ in range(r):
        # Choose a random c in the allowed range.
        max_c = ell // 2
        c = random.randint(0, max_c)
        a = T + c
        b = ell - 2*c
        # Randomly choose how many polarizaed spin-s particles to include - fixed number for all terms for consistency.
        n_gluons = random.randint(0, n_particles)
        n_gravitons = random.randint(n_gluons, n_particles) # choose a nunmber of gravitons from the remaining number of particles
        n_scalars = n_particles - n_gluons - n_gravitons # the rest are scalars. This number isn't used, but we keep it for clarity.
        term = generate_monomial(n_particles, n_gluons, n_gravitons, q)
        # Choose a sign; prepend "-" if negative, otherwise nothing.
        sign = random.choice([1, -1])
        if sign == -1:
            term = "-" + term
        # Otherwise (if positive) we leave it as is.
        numerator_terms.append(term)
    
    # Build the numerator as a sum.
    # We insert explicit " + " between terms if the next term is positive,
    # and " - " if the next term is negative (the sign is already embedded for negative terms).
    # For clarity we adjust the first term so that a leading "+" is not shown.
    combined_numerators = numerator_terms[0]
    for term in numerator_terms[1:]:
        if term.startswith("-"):
            combined_numerators += " - " + term[1:]  # remove the minus (already inserted)
        else:
            combined_numerators += " + " + term
    
    amplitude = f"(({combined_numerators}))/({common_den})"
    return amplitude



In [128]:
# Let's test this and generate a simple amplitude
amp = generate_amplitude(n_particles=4, r=4, q=4, ell=2)
print("Generated amplitude:")
print(amp)

Trying to make a monomial with 4 particles, 4 gluons, 4 gravitons, and mass dimension 4. Need 6 pp factors, 0 gluon e.p factors, 2 gluon e.e factors, 2 graviton e.p factors, and 1 graviton e.e factors.
[3, 2, 1, 4]
[6, 8, 5, 7]
Trying to make a monomial with 4 particles, 0 gluons, 1 gravitons, and mass dimension 4. Need 7 pp factors, 0 gluon e.p factors, 0 gluon e.e factors, 1 graviton e.p factors, and 0 graviton e.e factors.
[]
[1]
Trying to make a monomial with 4 particles, 1 gluons, 3 gravitons, and mass dimension 4. Need 5 pp factors, 1 gluon e.p factors, 0 gluon e.e factors, 1 graviton e.p factors, and 1 graviton e.e factors.
[1]
[3, 2, 4]
Trying to make a monomial with 4 particles, 2 gluons, 2 gravitons, and mass dimension 4. Need 8 pp factors, 0 gluon e.p factors, 1 gluon e.e factors, 0 graviton e.p factors, and 1 graviton e.e factors.
[1, 2]
[3, 4]
Generated amplitude:
((-p2.p3 * p2.p4 * p2.p3 * p1.p2 * p1.p3 * p3.p4 * e3.e2 * e1.e4 * e6.p1 * e6.p4 * e8.p2 * e8.p2 * e5.e7 * e5.

In [129]:
import re

def symbolic_scaling(amp):
    # Expect amp of the form: (( term1 ± term2 ± ... ))/( denominator )
    pattern = r'\(\(\s*(.*?)\s*\)\)/\(\s*(.*?)\s*\)'
    m = re.fullmatch(pattern, amp, re.DOTALL)
    if not m:
        raise ValueError("Amplitude string not in expected format: ((...))/(...)")
    num_str, den_str = m.groups()
    # Split numerator into terms (keep in mind terms may be prefixed with +/-).
    # This regex finds a term as an optional sign followed by a string that does not contain '+' or '-'
    term_strs = re.findall(r'[+-]?\s*([^+-]+)', num_str)
    term_scalings = []
    for term in term_strs:
        # Remove any extraneous whitespace.
        term = term.strip()
        # Count momentum factors (each p_i) and polarization factors (each e_i)
        np_term = len(re.findall(r'p\d+', term))
        ne_term = len(re.findall(r'e\d+', term))
        term_scalings.append((np_term, ne_term))
    # Ensure all numerator terms have the same scaling.
    first = term_scalings[0]
    if not all(ts == first for ts in term_scalings):
        print("Warning: Not all numerator terms have the same scaling!")
        for i, ts in enumerate(term_scalings):
            print(f"  Term {i+1}: m^({ts[0]}), l^({ts[1]})")
        raise ValueError("Inconsistent term scaling in numerator.")
    # Denominator should contain only momentum factors.
    np_den = len(re.findall(r'p\d+', den_str))
    # Overall scaling per term is m^(np_term - np_den) * l^(ne_term).
    final_m_exp = first[0] - np_den
    final_l_exp = first[1]
    return f"m^({final_m_exp}) * l^({final_l_exp})"

symbolic_scaling(amp)

  Term 1: m^(16), l^(12)
  Term 2: m^(16), l^(2)
  Term 3: m^(13), l^(7)
  Term 4: m^(16), l^(6)


ValueError: Inconsistent term scaling in numerator.